# 📡 Telecom Customer Churn Prediction Using Machine Learning

### B.Tech CSE — Machine Learning Case Study | 20 Marks

**Problem:** Predict whether a telecom customer is likely to churn using customer subscription, service, contract and billing attributes.

**Required algorithms**
1. Logistic Regression
2. KNN
3. Decision Tree
4. Random Forest
5. Gradient Boosting

**Required evaluation**
- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix
- K-fold cross-validation

> **Business emphasis:** Recall is particularly important because a false negative represents a customer who actually churns but was not identified.

This notebook is the **main academic submission**. The Streamlit application in `deployment/app.py` is the deployment component.


## 1. Dataset and Source

This project uses the **IBM Telco Customer Churn sample dataset**. IBM describes it as sample data for a fictional telecommunications company. The public IBM repository contains **7,043 customer records and 21 raw columns**.

The notebook downloads the CSV automatically from IBM's archived public GitHub repository if it is not already available locally.

**Source:** IBM, `IBM/telco-customer-churn-on-icp4d`

The dataset is a cross-sectional snapshot with a churn label; therefore this project is a **classification/retention-prioritization prototype**, not a time-series forecast of a particular future month.


In [ ]:
# 2. Install/import libraries
import sys, subprocess, os, urllib.request, warnings
warnings.filterwarnings("ignore")

packages = ["pandas", "numpy", "matplotlib", "seaborn", "scikit-learn", "joblib"]
for pkg in packages:
    try:
        __import__("sklearn" if pkg == "scikit-learn" else pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, roc_curve
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
print("Python:", sys.version.split()[0])
print("Libraries imported successfully.")


In [ ]:
# 3. Load the IBM dataset
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
LOCAL_PATH = "../data/Telco-Customer-Churn.csv"

if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    source_used = "Local copy"
else:
    df = pd.read_csv(DATA_URL)
    source_used = "IBM public GitHub raw URL"

print("Source:", source_used)
print("Shape:", df.shape)
display(df.head())


## 4. Understand the Dataset

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nTarget distribution:")
display(df["Churn"].value_counts())
display((df["Churn"].value_counts(normalize=True) * 100).round(2).rename("percentage"))


## 5. Data Quality and Missing Values

`TotalCharges` is stored as text in the raw dataset. Blank values must be converted to numeric missing values before modelling.

In [ ]:
# Convert TotalCharges to numeric; invalid/blank strings become NaN
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing values:")
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_count"))

print("Duplicate customer IDs:", df["customerID"].duplicated().sum())
print("Duplicate full rows:", df.duplicated().sum())


In [ ]:
# Show the exact rows affected by missing TotalCharges
missing_total = df[df["TotalCharges"].isna()]
print("Rows with missing TotalCharges:", len(missing_total))
display(missing_total[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]].head(15))


## 6. Exploratory Data Analysis (EDA)

The EDA addresses the case-study questions:
- Is churn related to contract type?
- How does tenure relate to churn?
- How do monthly charges relate to churn?
- Which service/payment categories show different churn rates?


In [ ]:
# Churn distribution
plt.figure(figsize=(6,4))
ax = sns.countplot(data=df, x="Churn")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.show()

print("Overall churn rate:", f"{(df['Churn']=='Yes').mean():.2%}")


In [ ]:
# Contract vs churn
contract_churn = pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100
display(contract_churn.round(2))

plt.figure(figsize=(8,5))
sns.barplot(data=df, x="Contract", y=(df["Churn"]=="Yes").astype(int), errorbar=None)
plt.title("Average Churn Rate by Contract Type")
plt.ylabel("Churn Rate")
plt.xlabel("Contract")
plt.gca().yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
plt.tight_layout()
plt.show()


In [ ]:
# Tenure vs churn
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="Churn", y="tenure")
plt.title("Tenure Distribution by Churn")
plt.xlabel("Churn")
plt.ylabel("Tenure (months)")
plt.tight_layout()
plt.show()

tenure_band = pd.cut(
    df["tenure"],
    bins=[-1, 6, 12, 24, 48, 72],
    labels=["0-6", "7-12", "13-24", "25-48", "49-72"]
)
tenure_churn = pd.crosstab(tenure_band, df["Churn"], normalize="index") * 100
display(tenure_churn.round(2))


In [ ]:
# Monthly charges vs churn
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges by Churn")
plt.tight_layout()
plt.show()

# Internet service and payment method
fig, axes = plt.subplots(1, 2, figsize=(13,5))
sns.barplot(
    data=df.assign(ChurnFlag=(df["Churn"]=="Yes").astype(int)),
    x="InternetService", y="ChurnFlag", errorbar=None, ax=axes[0]
)
axes[0].set_title("Churn Rate by Internet Service")
axes[0].set_ylabel("Churn Rate")

sns.barplot(
    data=df.assign(ChurnFlag=(df["Churn"]=="Yes").astype(int)),
    x="PaymentMethod", y="ChurnFlag", errorbar=None, ax=axes[1]
)
axes[1].set_title("Churn Rate by Payment Method")
axes[1].tick_params(axis="x", rotation=20)
axes[1].set_ylabel("Churn Rate")
plt.tight_layout()
plt.show()


### EDA interpretation

The plots should be interpreted as **associations**, not causal proof. In this dataset, month-to-month customers, shorter-tenure customers and some service/payment segments show visibly different churn rates. The exact percentages are calculated from the loaded IBM sample data.

This directly answers:
- **Do month-to-month contracts show a higher churn rate?** Compare the contract-rate table/plot.
- **Is tenure important?** Compare tenure distributions and tenure bands.


## 7. Prepare Features and Target

`customerID` is an identifier and is excluded from the model.

Target:
- `Churn = Yes` → 1
- `Churn = No` → 0

Numeric features:
- SeniorCitizen
- tenure
- MonthlyCharges
- TotalCharges

Categorical features:
- all remaining service/contract/payment attributes


In [ ]:
target = "Churn"
drop_cols = ["customerID", target]

X = df.drop(columns=drop_cols).copy()
y = (df[target] == "Yes").astype(int)

numeric_features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = [c for c in X.columns if c not in numeric_features]

print("Numeric features:", numeric_features)
print("Categorical feature count:", len(categorical_features))
print("X shape:", X.shape)
print("y shape:", y.shape)


## 8. Preprocessing Pipeline

The preprocessing is placed inside a scikit-learn `Pipeline` so that training and unseen-customer prediction use the same transformations.

- Numeric missing values → median imputation
- Categorical missing values → most-frequent imputation
- Categorical values → one-hot encoding
- Numeric values → standardization


In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessing pipeline created.")


## 9. Train/Test Split

An 80/20 **stratified** split is used so the churn/non-churn class proportion is approximately preserved in both sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training churn rate:", f"{y_train.mean():.2%}")
print("Test churn rate:", f"{y_test.mean():.2%}")


## 10. Train the Five Required Classification Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "KNN": KNeighborsClassifier(n_neighbors=15),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=10, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=4,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=180, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE
    )
}

pipelines = {
    name: Pipeline([("preprocess", preprocessor), ("model", model)])
    for name, model in models.items()
}

print("Models:", ", ".join(pipelines.keys()))


## 11. Test-Set Evaluation

We calculate:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

**Recall is the primary business metric** for this case study.


In [ ]:
test_rows = []
test_predictions = {}
test_probabilities = {}

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]

    test_predictions[name] = pred
    test_probabilities[name] = prob

    test_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-Score": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

test_results = pd.DataFrame(test_rows).sort_values(
    ["Recall", "F1-Score"], ascending=False
).reset_index(drop=True)

display(test_results.style.format({
    "Accuracy": "{:.3%}",
    "Precision": "{:.3%}",
    "Recall": "{:.3%}",
    "F1-Score": "{:.3%}",
    "ROC-AUC": "{:.3%}"
}))


In [ ]:
# Metric comparison chart
plot_metrics = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
ax = test_results.set_index("Model")[plot_metrics].plot(
    kind="bar", figsize=(12,6), ylim=(0,1)
)
plt.title("Test-Set Model Comparison")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()


## 12. Confusion Matrices

A confusion matrix contains:
- **TN:** correctly predicted non-churn
- **TP:** correctly predicted churn
- **FP:** predicted churn but customer did not churn
- **FN:** predicted non-churn but customer actually churned

For this business problem, **FN is especially important** because it represents a missed churner.


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20,4))
for ax, (name, pred) in zip(axes, test_predictions.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()


## 13. ROC Curves

ROC-AUC is included as an additional evaluation metric. It measures how well the model ranks churners above non-churners across classification thresholds.


In [ ]:
plt.figure(figsize=(9,6))
for name, prob in test_probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
plt.plot([0,1], [0,1], "--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.tight_layout()
plt.show()


## 14. Stratified 5-Fold Cross-Validation

Cross-validation checks whether the model's performance is reasonably stable across different splits.

We report mean:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_rows = []
for name, pipe in pipelines.items():
    scores = cross_validate(
        pipe, X, y,
        cv=cv,
        scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
        n_jobs=-1
    )
    cv_rows.append({
        "Model": name,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1-Score": scores["test_f1"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean()
    })

cv_results = pd.DataFrame(cv_rows).sort_values(
    ["CV Recall", "CV F1-Score"], ascending=False
).reset_index(drop=True)

display(cv_results.style.format({
    "CV Accuracy": "{:.3%}",
    "CV Precision": "{:.3%}",
    "CV Recall": "{:.3%}",
    "CV F1-Score": "{:.3%}",
    "CV ROC-AUC": "{:.3%}"
}))


## 15. Model Selection

The case study specifically says to emphasize Recall. Therefore:

**Primary criterion:** highest test-set Recall.

**Secondary criterion:** F1-score, followed by cross-validation recall, to avoid selecting a model only because of one metric.

This is a documented selection rule, not an arbitrary preference.


In [ ]:
best_model_name = test_results.iloc[0]["Model"]
best_pipeline = pipelines[best_model_name]

print("Selected model:", best_model_name)
print("Reason: highest test-set Recall under the case-study selection priority.")
display(test_results.iloc[[0]])


## 16. Feature Analysis / Key Churn Drivers

Feature analysis helps answer:
- Which attributes are most associated with churn?
- Is tenure or contract type more important?

For Logistic Regression, absolute coefficients indicate the strength/direction of the model's linear association after preprocessing. For tree models, feature importance can be used.

**Important:** feature importance/coefficients are predictive relationships, not proof of causation.


In [ ]:
# Fit a logistic model specifically for interpretable coefficient analysis.
log_pipe = pipelines["Logistic Regression"]
log_pipe.fit(X_train, y_train)

feature_names = log_pipe.named_steps["preprocess"].get_feature_names_out()
coef = log_pipe.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coef,
    "AbsCoefficient": np.abs(coef)
}).sort_values("AbsCoefficient", ascending=False)

display(coef_df.head(20))

top = coef_df.head(15).sort_values("Coefficient")
plt.figure(figsize=(10,6))
plt.barh(top["Feature"], top["Coefficient"])
plt.axvline(0, linewidth=1)
plt.title("Top Logistic Regression Feature Coefficients")
plt.xlabel("Coefficient")
plt.tight_layout()
plt.show()


In [ ]:
# Direct descriptive analysis of contract and tenure.
contract_rates = (
    df.assign(ChurnFlag=(df["Churn"]=="Yes").astype(int))
      .groupby("Contract")["ChurnFlag"].mean()
      .sort_values(ascending=False)
)

tenure_rates = (
    df.assign(
        ChurnFlag=(df["Churn"]=="Yes").astype(int),
        TenureBand=tenure_band
    )
    .groupby("TenureBand", observed=False)["ChurnFlag"]
    .mean()
)

print("Churn rate by contract:")
display((contract_rates * 100).round(2).rename("Churn %"))

print("Churn rate by tenure band:")
display((tenure_rates * 100).round(2).rename("Churn %"))


## 17. Final Answers to the Case-Study Questions

1. **Can customer churn be predicted from subscription data?**  
   Yes. The models achieve predictive performance above a random baseline on the held-out test set, although performance depends on the chosen metric and threshold.

2. **Which service attributes are most associated with churn?**  
   Use the EDA tables and feature-analysis output above. Contract, tenure, monthly charges, internet/payment/service attributes can show meaningful predictive associations.

3. **Which algorithm has the highest recall?**  
   The test-results table above identifies it; the selection rule prioritizes this model.

4. **Do month-to-month contracts show a higher churn rate?**  
   The contract churn-rate table directly answers this for the IBM sample.

5. **What proportion of churning customers does the model detect?**  
   This is the selected model's Recall.

6. **How well does the model perform on unseen customers?**  
   The held-out test-set metrics and 5-fold CV provide evidence.

7. **Can the model be deployed as a retention-support application?**  
   Yes. The accompanying Streamlit app loads the same dataset, trains the same model-selection workflow, and provides an interactive prediction interface.


## 18. Limitations and Real-World Applicability

- The IBM data is sample data for a fictional telecom company.
- It is a cross-sectional snapshot rather than a sequence of future monthly observations.
- Historical associations do not prove causal effects.
- Customer behavior can change over time (data drift).
- A probability is an estimate, not a guarantee.
- Production deployment requires calibration, monitoring, privacy controls, fairness checks and business validation.
- A retention offer should not be automatically triggered solely from this model without appropriate business rules.


In [ ]:
# 19. Save reproducible results for the deployment/report
os.makedirs("../results", exist_ok=True)

test_results.to_csv("../results/model_comparison_test_set.csv", index=False)
cv_results.to_csv("../results/model_comparison_5fold_cv.csv", index=False)
coef_df.to_csv("../results/logistic_feature_coefficients.csv", index=False)

# Save the full dataset locally when possible so the deployment can reuse it.
try:
    df.to_csv("../data/Telco-Customer-Churn.csv", index=False)
    print("Saved local dataset copy to ../data/Telco-Customer-Churn.csv")
except Exception as e:
    print("Could not save local dataset copy:", e)

print("Results exported.")


# 20. Deployment Handoff

The separate `deployment/app.py` is designed to:

- load the IBM dataset
- perform the same preprocessing
- train the five required models
- select the model using the same recall-priority rule
- provide a prediction form
- show High/Low risk and probability
- explain which customer characteristics contributed to the prediction
- provide demo profiles so the application can reliably demonstrate both higher- and lower-risk cases

Run it from the project root with:

```bash
python3 -m pip install -r deployment/requirements.txt
python3 -m streamlit run deployment/app.py
```
